# **Test OpenRouter Provider Selection**

In [557]:
from openai import OpenAI

from mdner_llm.common import load_api_key

api_key = load_api_key("OPENROUTER_API_KEY")
# Instantiate an OpenAI client for the requested model
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [558]:
# Define arguments for the API call
kwargs = {
    "messages": [
        {"role": "system", "content": "Extract entities as structured JSON."},
        {
            "role": "user",
            "content": "Extract entities from the following text:"
            "GAFF is a widely used force field in molecular dynamics simulations. AutoDock is a popular software for molecular docking studies. The simulation was run for 20 ns.",
        },
    ],
}

In [559]:
# Specify provider selection criteria
from mdner_llm.models.entities import ListOfEntities


provider = "wandb/fp8"
if provider:
    kwargs["extra_body"] = {
        "provider": {"order": [provider], "allow_fallbacks": False}
    }

kwargs["response_model"] = ListOfEntities
kwargs["max_retries"] = 3

In [560]:
kwargs


{'messages': [{'role': 'system',
   'content': 'Extract entities as structured JSON.'},
  {'role': 'user',
   'content': 'Extract entities from the following text:GAFF is a widely used force field in molecular dynamics simulations. AutoDock is a popular software for molecular docking studies. The simulation was run for 20 ns.'}],
 'extra_body': {'provider': {'order': ['wandb/fp8'],
   'allow_fallbacks': False}},
 'response_model': mdner_llm.models.entities.ListOfEntities,
 'max_retries': 3}

In [561]:
import time

import instructor

client = instructor.from_provider(
    "openrouter/qwen/qwen3.6-27b",
    async_client=False,
    mode=instructor.Mode.JSON,
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)
# Query the LLM
start_time = time.perf_counter()
llm_response, completion = client.create_with_completion(**kwargs)
llm_response

ListOfEntities(entities=[ForceField(category='FFM', text='GAFF'), SoftwareName(category='SOFTNAME', text='AutoDock'), SimulationTime(category='STIME', text='20 ns')])

In [562]:
completion

ChatCompletion(id='gen-1781646625-RLEcaVzpd5FZukgMcBsF', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "entities": [\n    {\n      "text": "GAFF",\n      "category": "FFM"\n    },\n    {\n      "text": "AutoDock",\n      "category": "SOFTNAME"\n    },\n    {\n      "text": "20 ns",\n      "category": "STIME"\n    }\n  ]\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Input Text:** "GAFF is a widely used force field in molecular dynamics simulations. AutoDock is a popular software for molecular docking studies. The simulation was run for 20 ns."\n   - **Task:** Extract entities and return them as a JSON object matching the provided schema.\n   - **Schema Definitions:**\n     - `ForceField` (FFM): force fields used in MD simulations\n     - `Molecule` (MOL): molecules, proteins, lipids, peptides\n

In [563]:
# Display the only information about the provider used for the response
completion.provider


'WandB'

In [564]:
completion.usage


CompletionUsage(completion_tokens=1667, prompt_tokens=1285, total_tokens=2952, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=1495, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=784), cost=0.00639588, is_byok=False, cost_details={'upstream_inference_cost': 0.00639588, 'upstream_inference_prompt_cost': 0.00039468, 'upstream_inference_completions_cost': 0.0060012})

In [565]:
# With no framework
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

resp = client.chat.completions.create(
    model="google/gemma-4-31b-it",
    messages=[{"role": "user", "content": "hello"}],
    extra_body={
        "provider": {
            "order": ["wandb/bf16"],
            "allow_fallbacks": False,
        }
    },
)

resp

ChatCompletion(id='gen-1781646643-vaXPIQqxKBWG0ilVpn51', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I help you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1781646643, model='google/gemma-4-31b-it-20260402', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=10, prompt_tokens=14, total_tokens=24, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=5.18e-06, is_byok=False, cost_details={'upstream_inference_cost': 5.18e-06, 'upstream_inference_prompt_cost': 1.68e-06, 'upstream_inference_completions_cost': 3.5e

------
Example that doesn't work : Using the `instructor` framework with the `openrouter/google/gemma-4-31b-it` model and the provider `wandb/bf16` (which is within the provider choices for this model : https://openrouter.ai/google/gemma-4-31b-it?endpoint=c8d581d3-6a0a-480b-8cf0-cf010bef1a70#providers). The code below will raise an error.

> Note: It works when response_model is set to `None`.

In [572]:
provider = "wandb/bf16"
if provider:
    kwargs["extra_body"] = {"provider": {"order": [provider], "allow_fallbacks": False}}

kwargs["response_model"] = ListOfEntities
# Or Without a response model
# kwargs["response_model"] = None
kwargs["max_retries"] = 3
kwargs

{'messages': [{'role': 'system',
   'content': 'Extract entities as structured JSON.\n\n\n        Parse the content and return a JSON object matching this schema:\n\n        {\n  "$defs": {\n    "ForceField": {\n      "description": "Entity representing a force field used in the MD simulation.",\n      "properties": {\n        "category": {\n          "const": "FFM",\n          "default": "FFM",\n          "description": "Category for force field or model entities.",\n          "title": "Category",\n          "type": "string"\n        },\n        "text": {\n          "description": "Extracted text content.",\n          "title": "Text",\n          "type": "string"\n        }\n      },\n      "required": [\n        "text"\n      ],\n      "title": "ForceField",\n      "type": "object"\n    },\n    "Molecule": {\n      "description": "Entity representing a molecule, protein, lipid, peptide, or similar object.",\n      "properties": {\n        "category": {\n          "const": "MOL",\n    

In [ ]:
client = instructor.from_provider(
    "openrouter/google/gemma-4-31b-it",
    async_client=False,
    mode=instructor.Mode.JSON,
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)
# Query the LLM
start_time = time.perf_counter()
llm_response, completion = client.create_with_completion(**kwargs)
llm_response


InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    Error code: 404 - {'error': {'message': 'No endpoints found for google/gemma-4-31b-it.', 'code': 404}, 'user_id': 'org_2xzaeQPJTBeOmEa2Twil90EHNf3'}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    Error code: 404 - {'error': {'message': 'No endpoints found for google/gemma-4-31b-it.', 'code': 404}, 'user_id': 'org_2xzaeQPJTBeOmEa2Twil90EHNf3'}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="3">
<exception>
    Error code: 404 - {'error': {'message': 'No endpoints found for google/gemma-4-31b-it.', 'code': 404}, 'user_id': 'org_2xzaeQPJTBeOmEa2Twil90EHNf3'}
</exception>
<completion>
    None
</completion>
</generation>

</failed_attempts>

<last_exception>
    Error code: 404 - {'error': {'message': 'No endpoints found for google/gemma-4-31b-it.', 'code': 404}, 'user_id': 'org_2xzaeQPJTBeOmEa2Twil90EHNf3'}
</last

In [ ]:
# To run only if response_model is set to None, otherwise it will raise an error
llm_response.provider